# Evaluate Retrieval, Prompting, and Routing for *Clinical Synopsis: Source-Grounded Patient Summaries*

This notebook evaluates how well the Clinical Synopsis RAG pipeline retrieves and answers patient-specific clinical questions.

It does four things:

1. Builds a small ground-truth evaluation set by selecting 9 patients across complexity buckets and defining gold chunks for 4 question types:
   - patient overview
   - conditions
   - medications
   - oncology timeline
2. Measures retrieval quality across lexical, semantic, and hybrid search using ranking metrics (Hit@K and MRR@K).
3. Tests lexical boost configurations to see whether field weighting (title/heading/chunk text) meaningfully changes retrieval outcomes.
4. Evaluates generation quality with LLM-as-judge in two ways:
   - context-grounded judging (`evaluate_relevance`)
   - reference-based judging against gold summaries (`evaluate_against_reference`)

The final section compares baseline vs **question-routed prompting**, showing how task-specific instructions can improve clinical answer quality even when retrieval metrics are similar.

## Contents

- [Ground truth](#ground-truth)
   - [Pick 9 patients with different complexity scores](#pick-9-patients-with-different-complexity-scores)
   - [Select gold chunks](#select-gold-chunks)
- [Evaluate retrieval (lexical, semantic, hybrid)](#evaluate-retrieval-lexical-semantic-hybrid)
   - [Evaluate retrieval with different boost values](#evaluate-retrieval-with-different-boost-values)
- [Evaluate prompt](#evaluate-prompt)
   - [Baseline RAG Pipeline](#baseline-rag-pipeline)
      - [LLM-as-judge evaluating against a reference summary](#llm-as-judge-evaluating-against-a-reference-summary)
      - [Overview question](#overview-question)
      - [Conditions question](#conditions-question)
      - [Conclusion](#conclusion)
   - [Question-Routed RAG Pipeline](#question-routed-rag-pipeline)
      - [Example patient with question routing](#example-patient-with-question-routing)
      - [Example patient without question routing](#example-patient-without-question-routing)

In [1]:
from pathlib import Path
import sys

project_root = Path.cwd().parent
sys.path.insert(0, str(project_root / "clinical_synopsis"))

# Ground truth

- We pick a representative set of 9 patients (out of a total of 50), covering a range of complexities of life-time health records including oncology history. 
- We define a set of 4 questions that reflect 
    - the goals of the Clinical Synopsis app: 
    (1) patient overview, 
    (2) conditions, 
    (3) medications, 
    (4) oncology history. 
    - and which cover the range of documents available for each patients:
    'patient_overview.md', 'oncology_timeline.md', 'procedures.csv', 'diagnostic_reports.csv', 'oncology_timeline_events.csv', 'observations.csv', 'conditions.csv', 'encounters.csv',  'medications.csv'.

- We chose gold chunks for each patient for each question:
    - for summary questions, we use all oncology-timeline and selected conditions chunks as gold.
    .....
    - for **more specific questions** (e.g., “What chemotherapy regimen?”), define a narrower gold set.


Each chunk in `patient_overview_gold` is selected based on a rule: it comes from a conditions document, or has a heading of Recent Condition, Recent Results, or Procedures in the overview document.

Each chunk in `conditions_gold` comes from a conditions document, has a heading of Recent Condition, or Recent Results in the overview document.

Each chunk in `medications_gold` comes from the overview document and has a heading Medications.

Each chunk in `oncology_timeline_gold` comes fomr the oncology_timeline document.

## Pick 9 patients with different complexity scores

In [2]:
import pandas as pd

manifest_path = (
    project_root / "data" / "processed" / "mcode_breast_sample_50_manifest.csv"
)

# Load manifest
df = pd.read_csv(manifest_path)

print("Shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

Shape: (50, 17)

Columns:
['filename', 'patient_id', 'patient_name', 'n_resources', 'n_encounters', 'n_observations', 'n_conditions', 'n_procedures', 'n_medication_requests', 'n_medication_administrations', 'n_diagnostic_reports', 'first_date', 'last_date', 'followup_days', 'complexity_score', 'complexity_bucket', 'sample_seed']


In [3]:
# Check bucket distribution
print("Bucket counts in full set of 50 patients:")
print(df["complexity_bucket"].value_counts())

# Sample 3 patients from each bucket (low, medium, high)
bucket_targets = {"low": 3, "medium": 3, "high": 3}
selected_rows = []

for bucket, n in bucket_targets.items():
    bucket_df = df[df["complexity_bucket"] == bucket].copy()
    if len(bucket_df) < n:
        raise ValueError(f"Not enough patients in bucket '{bucket}' to sample {n}.")

    # Random sample with a fixed seed for reproducibility
    sampled_bucket = bucket_df.sample(n=n, random_state=42)
    selected_rows.append(sampled_bucket)

selected_df = pd.concat(selected_rows).reset_index(drop=True)

print("\nSelected 9 patients (3 per bucket):")
display(
    selected_df[
        [
            "patient_id",
            "patient_name",
            "complexity_bucket",
            "n_resources",
            "complexity_score",
        ]
    ]
)

# Just the list of patient_ids for later use
selected_patient_ids = selected_df["patient_id"].tolist()

Bucket counts in full set of 50 patients:
complexity_bucket
low       17
high      17
medium    16
Name: count, dtype: int64

Selected 9 patients (3 per bucket):


,patient_id,patient_name,complexity_bucket,n_resources,complexity_score
0,d65197b3-056a-2136-b584-77f43c29da3f,Corrie32 Boyle917,low,230,317
1,f3739580-797d-ae04-eebf-aeddb2fc2f64,Florine959 Stark857,low,261,330
2,4736727e-63f4-071a-1516-a49310f5a052,Mónica985 Serrato62,low,436,602
3,29f6beee-162f-0113-7884-72245814693f,Eula461 Crooks415,medium,1854,2786
4,41681ed6-efc5-94c0-1bc0-f60b34dbd31b,Beth967 Cremin516,medium,1843,2800
5,aee216e6-cbe8-eaf2-3241-4bd1e8a01494,Deeann517 Torp761,medium,2191,3340
6,ecc4a7d0-8838-36b4-44ba-676d5a1f7927,Francina926 Von197,high,2945,4458
7,3a1c7c7b-0f87-e7ba-f2ed-1b0882fe3678,Rosetta750 Stroman228,high,3055,4536
8,f203e11d-5573-1624-69b8-af8436987b3e,Shawana711 Lakin515,high,3365,4812


In [4]:
selected_patient_ids

['d65197b3-056a-2136-b584-77f43c29da3f',
 'f3739580-797d-ae04-eebf-aeddb2fc2f64',
 '4736727e-63f4-071a-1516-a49310f5a052',
 '29f6beee-162f-0113-7884-72245814693f',
 '41681ed6-efc5-94c0-1bc0-f60b34dbd31b',
 'aee216e6-cbe8-eaf2-3241-4bd1e8a01494',
 'ecc4a7d0-8838-36b4-44ba-676d5a1f7927',
 '3a1c7c7b-0f87-e7ba-f2ed-1b0882fe3678',
 'f203e11d-5573-1624-69b8-af8436987b3e']

In [5]:
# load chunks_df and onc_chunks

import sqlite3
import pandas as pd
from pathlib import Path

db_path = Path("../data/retrieval/metadata.db")
patient_ids = selected_patient_ids

conn = sqlite3.connect(db_path)

# Multiple patient_ids: build an IN (...) placeholder list.
if not patient_ids:
    chunks_df = pd.DataFrame()  # avoid invalid SQL: IN ()
else:
    placeholders = ",".join(["?"] * len(patient_ids))
    query = f"""
    SELECT
        chunks.patient_id,
        documents.doc_type,
        documents.title,
        chunks.heading,
        chunks.chunk_id,
        chunks.is_oncology,
        chunks.chunk_text
    FROM chunks
    JOIN documents ON chunks.document_id = documents.document_id
    WHERE chunks.patient_id IN ({placeholders})
    """
    chunks_df = pd.read_sql_query(query, conn, params=patient_ids)

conn.close()

# display(chunks_df.head())
print("Number of all chunks for selected patients:", len(chunks_df))

onc_chunks = chunks_df[chunks_df["is_oncology"] == 1]
# display(onc_chunks.head())
print("Number of oncology chunks for selected patients:", len(onc_chunks))

Number of all chunks for selected patients: 14390
Number of oncology chunks for selected patients: 890


## Select gold chunks

In [6]:
# import pandas as pd
# from IPython.display import display

# QUESTION_TYPE = "patient_overview"
# QUESTION_TEXT = "Give a concise overview of this patient’s medical background and current care context."

# overview_doc_types = ["conditions"]
# overview_headings = [
#     "Recent Condition",
#     "Recent Results",
#     "Procedures",
# ]

# # Boolean mask for relevant chunks according to your rule
# mask_conditions = chunks_df["doc_type"] == "conditions"
# mask_headings = chunks_df["heading"].isin(overview_headings)

# relevant_mask = mask_conditions | mask_headings

# patient_overview_gold = (
#     chunks_df.loc[relevant_mask, ["patient_id", "chunk_id"]]
#     .groupby("patient_id")["chunk_id"]
#     .apply(list)
#     .reset_index()
#     .rename(columns={"chunk_id": "gold_chunk_ids"})
# )

# patient_overview_gold["question_type"] = QUESTION_TYPE
# patient_overview_gold["question_text"] = QUESTION_TEXT

# # Optional: reorder columns for clarity
# patient_overview_gold = patient_overview_gold[
#     ["patient_id", "question_type", "question_text", "gold_chunk_ids"]
# ]

# display(patient_overview_gold)

In [7]:
import pandas as pd
from IPython.display import display

QUESTION_TYPE = "patient_overview"
QUESTION_TEXT = "Give a concise overview of this patient’s medical background and current care context."

overview_headings = [
    "Recent Condition",
    "Recent Results",
    "Procedures",
]

# Summary-like chunks from patient_overview
mask_po_headings = (chunks_df["doc_type"] == "patient_overview") & (
    chunks_df["heading"].isin(overview_headings)
)

# Optional: include conditions.csv as secondary context
mask_conditions = chunks_df["doc_type"] == "conditions"

relevant_mask = mask_po_headings  #| mask_conditions  # or just mask_po_headings if you want it narrower

patient_overview_gold = (
    chunks_df.loc[relevant_mask, ["patient_id", "chunk_id"]]
    .groupby("patient_id")["chunk_id"]
    .apply(list)
    .reset_index()
    .rename(columns={"chunk_id": "gold_chunk_ids"})
)

patient_overview_gold["question_type"] = QUESTION_TYPE
patient_overview_gold["question_text"] = QUESTION_TEXT

patient_overview_gold = patient_overview_gold[
    ["patient_id", "question_type", "question_text", "gold_chunk_ids"]
]

display(patient_overview_gold)

,patient_id,question_type,question_text,gold_chunk_ids
0,29f6beee-162f-0113-7884-72245814693f,patient_overview,Give a concise overview of this patient’s medi...,"[abc678e3f0fdb2313c03b92ff62bf06f08120f7b, c1b..."
1,3a1c7c7b-0f87-e7ba-f2ed-1b0882fe3678,patient_overview,Give a concise overview of this patient’s medi...,"[a970cc1ae6abca57a8813b4cc2aea2df1d41cde1, 519..."
2,41681ed6-efc5-94c0-1bc0-f60b34dbd31b,patient_overview,Give a concise overview of this patient’s medi...,"[0590211551c10b6259f5a9062003667e93e8f4e6, 9c7..."
3,4736727e-63f4-071a-1516-a49310f5a052,patient_overview,Give a concise overview of this patient’s medi...,"[63f01e07f0f326ce090ed1d0e1e9cb9ddb88d53b, 49c..."
4,aee216e6-cbe8-eaf2-3241-4bd1e8a01494,patient_overview,Give a concise overview of this patient’s medi...,"[aff56cb2f4bb43dff398ad5e744d1f5a364b7670, 280..."
5,d65197b3-056a-2136-b584-77f43c29da3f,patient_overview,Give a concise overview of this patient’s medi...,"[abc3ffd40cbb13d210ac910a0f5aaa43b998a587, e84..."
6,ecc4a7d0-8838-36b4-44ba-676d5a1f7927,patient_overview,Give a concise overview of this patient’s medi...,"[aee14674ce11ab5daa05de7c09f81db1f25365a4, 95b..."
7,f203e11d-5573-1624-69b8-af8436987b3e,patient_overview,Give a concise overview of this patient’s medi...,"[1f9771cf6cdb8353d65819a133ea34b11b3b8e58, 78c..."
8,f3739580-797d-ae04-eebf-aeddb2fc2f64,patient_overview,Give a concise overview of this patient’s medi...,"[a9165d0158523ea3b0c882ba42cd1f37f02f067d, c59..."


In [9]:
QUESTION_TYPE = "conditions"
QUESTION_TEXT = "What are the patient’s main diagnosed conditions?"

condition_headings = ["Recent Conditions", "Recent Results"]

mask_po_conditions = (  # only the patient_overview chunks with conditions
    (chunks_df["doc_type"] == "patient_overview")
    & (chunks_df["heading"].isin(condition_headings))
)

# For now, focus gold on patient_overview only
relevant_mask = mask_po_conditions

conditions_gold = (
    chunks_df.loc[relevant_mask, ["patient_id", "chunk_id"]]
    .groupby("patient_id")["chunk_id"]
    .apply(list)
    .reset_index()
    .rename(columns={"chunk_id": "gold_chunk_ids"})
)

conditions_gold["question_type"] = QUESTION_TYPE
conditions_gold["question_text"] = QUESTION_TEXT

conditions_gold = conditions_gold[
    ["patient_id", "question_type", "question_text", "gold_chunk_ids"]
]

display(conditions_gold)

,patient_id,question_type,question_text,gold_chunk_ids
0,29f6beee-162f-0113-7884-72245814693f,conditions,What are the patient’s main diagnosed conditions?,"[14326c002878a932e87427ea2a9ba2b6e6ad1404, abc..."
1,3a1c7c7b-0f87-e7ba-f2ed-1b0882fe3678,conditions,What are the patient’s main diagnosed conditions?,"[772ce44ecf1664ce3cd0f321962e31b1afeaf658, a97..."
2,41681ed6-efc5-94c0-1bc0-f60b34dbd31b,conditions,What are the patient’s main diagnosed conditions?,"[8123905ce9ef5c2a100a0400f86029469183dbb2, 059..."
3,4736727e-63f4-071a-1516-a49310f5a052,conditions,What are the patient’s main diagnosed conditions?,"[7ac2bda6270220cb79ace629a38ef8b5a13ddf31, 63f..."
4,aee216e6-cbe8-eaf2-3241-4bd1e8a01494,conditions,What are the patient’s main diagnosed conditions?,"[e733276d7db1b542f556b25839e3602be1339cbc, aff..."
5,d65197b3-056a-2136-b584-77f43c29da3f,conditions,What are the patient’s main diagnosed conditions?,"[45b0f62c2348b66d770620e4e5acfe630cf41f96, abc..."
6,ecc4a7d0-8838-36b4-44ba-676d5a1f7927,conditions,What are the patient’s main diagnosed conditions?,"[05d547f22443d60a0d31fa1776bed869da7459cd, aee..."
7,f203e11d-5573-1624-69b8-af8436987b3e,conditions,What are the patient’s main diagnosed conditions?,"[f73c3623d5df217075882e25106a9d6388dde40e, 1f9..."
8,f3739580-797d-ae04-eebf-aeddb2fc2f64,conditions,What are the patient’s main diagnosed conditions?,"[4c17e4784d257f85e9fd1ddd74054e5dc3adae9f, a91..."


In [11]:
QUESTION_TYPE = "medications"
QUESTION_TEXT = "What medications is the patient taking or has recently taken?"

med_doc_types = ["medications"]
med_headings = ["Medications"]  # adjust to exact heading text in patient_overview.md

mask_po_medications = (  # only the patient_overview chunks with medications
    (chunks_df["doc_type"] == "patient_overview")
    & (chunks_df["heading"].isin(med_headings))
)

# For now, focus gold on patient_overview only
relevant_mask = mask_po_medications

medications_gold = (
    chunks_df.loc[relevant_mask, ["patient_id", "chunk_id"]]
    .groupby("patient_id")["chunk_id"]
    .apply(list)
    .reset_index()
    .rename(columns={"chunk_id": "gold_chunk_ids"})
)

medications_gold["question_type"] = QUESTION_TYPE
medications_gold["question_text"] = QUESTION_TEXT

medications_gold = medications_gold[
    ["patient_id", "question_type", "question_text", "gold_chunk_ids"]
]

display(medications_gold)

,patient_id,question_type,question_text,gold_chunk_ids
0,29f6beee-162f-0113-7884-72245814693f,medications,What medications is the patient taking or has ...,[313381cf2c6383b0d66557ee95469ebb92cf7cc1]
1,3a1c7c7b-0f87-e7ba-f2ed-1b0882fe3678,medications,What medications is the patient taking or has ...,[3f1c94dc52eb9d6748171ca99d8d35ee5b0cc2cd]
2,41681ed6-efc5-94c0-1bc0-f60b34dbd31b,medications,What medications is the patient taking or has ...,[392065e296a5586b78eb9e2a43d08df5220cb956]
3,4736727e-63f4-071a-1516-a49310f5a052,medications,What medications is the patient taking or has ...,[7d4fb354f104d3cc11273d15182d3f548ca4031b]
4,aee216e6-cbe8-eaf2-3241-4bd1e8a01494,medications,What medications is the patient taking or has ...,[3b6a76df9116342cca819342b5a77a629c04d636]
5,d65197b3-056a-2136-b584-77f43c29da3f,medications,What medications is the patient taking or has ...,[24c7158a723f3ec52139411b806ca04fe110024e]
6,ecc4a7d0-8838-36b4-44ba-676d5a1f7927,medications,What medications is the patient taking or has ...,[e3fbadc0fd8c550e6501a326f6222e5c97538c8d]
7,f203e11d-5573-1624-69b8-af8436987b3e,medications,What medications is the patient taking or has ...,[e0893f6f1f540051fea4cc26cef7068cb59b41c6]
8,f3739580-797d-ae04-eebf-aeddb2fc2f64,medications,What medications is the patient taking or has ...,[be283721d4470a4515d90787d7ce60f5a92db51e]


In [13]:
QUESTION_TYPE = "oncology_timeline"
QUESTION_TEXT = "Summarize the patient’s oncology-related timeline, including major events and treatments."

timeline_doc_types = ["oncology_timeline"]

mask_timeline = chunks_df["doc_type"].isin(timeline_doc_types)

oncology_timeline_gold = (
    chunks_df.loc[mask_timeline, ["patient_id", "chunk_id"]]
    .groupby("patient_id")["chunk_id"]
    .apply(list)
    .reset_index()
    .rename(columns={"chunk_id": "gold_chunk_ids"})
)

oncology_timeline_gold["question_type"] = QUESTION_TYPE
oncology_timeline_gold["question_text"] = QUESTION_TEXT

oncology_timeline_gold = oncology_timeline_gold[
    ["patient_id", "question_type", "question_text", "gold_chunk_ids"]
]

display(oncology_timeline_gold)

,patient_id,question_type,question_text,gold_chunk_ids
0,29f6beee-162f-0113-7884-72245814693f,oncology_timeline,Summarize the patient’s oncology-related timel...,"[c21e06064148b6da08db071c8263d1da295ed93c, 17f..."
1,3a1c7c7b-0f87-e7ba-f2ed-1b0882fe3678,oncology_timeline,Summarize the patient’s oncology-related timel...,"[22d46b9aab128e2a1acdc0d26d872f8bd10f7f79, 674..."
2,41681ed6-efc5-94c0-1bc0-f60b34dbd31b,oncology_timeline,Summarize the patient’s oncology-related timel...,"[e5ca0cdb0930f81b8adfb7fa94b2b6cb38539862, 441..."
3,4736727e-63f4-071a-1516-a49310f5a052,oncology_timeline,Summarize the patient’s oncology-related timel...,"[5548c8e86138b4a33e1d8a12c518e43dcd667f50, 441..."
4,aee216e6-cbe8-eaf2-3241-4bd1e8a01494,oncology_timeline,Summarize the patient’s oncology-related timel...,"[30263ea4c834c2be4745a107b2f8386cbd98a1e4, 2fe..."
5,d65197b3-056a-2136-b584-77f43c29da3f,oncology_timeline,Summarize the patient’s oncology-related timel...,"[0ab557068bb85b16a4aad4d40f2a6c83cd45d5b3, 1fb..."
6,ecc4a7d0-8838-36b4-44ba-676d5a1f7927,oncology_timeline,Summarize the patient’s oncology-related timel...,"[4105225d21ffe94cb84279be1b62e59149f4ed35, fde..."
7,f203e11d-5573-1624-69b8-af8436987b3e,oncology_timeline,Summarize the patient’s oncology-related timel...,"[c9c3776fbdbf55e34536d8b71d32ba80b80f0ad2, 98d..."
8,f3739580-797d-ae04-eebf-aeddb2fc2f64,oncology_timeline,Summarize the patient’s oncology-related timel...,"[d963a6518333e3e87c905fcecd599bf3f90c3f21, 330..."


# Evaluate retrieval (lexical, semantic, hybrid)

We see that:
- Patient Overview favors semantic or hybrid retrieval, over lexical retrieval.

- For Conditions questions when the gold chunks included all the chunks from conditions.csv lexical search retrieved all of those chunks. BUt when gold chunks include only the specific summary chunk from patient_overview, lexical search isn’t retrieving that chunk at all in top‑5; semantic partially does; hybrid gets mixed results. 

- For Medication questions the situation is the same. Lexical search doesn’t retrieve the patient_overview medications summary chunk in the top‑5, for any patient. But semantic search always retrieves the summary chunk in top‑5 and the summary chunk is usually at rank 1–2. With hybrid search it is lower.

- For Oncology timeline questions lexical and hybrid search both perform well.

This suggests that the RAG pipeline would benefit from a task-specific retrieval strategy. However, in the next section we see that adding task-specific prompt instructions does the trick and hybrid search can be used overall.


In [14]:
import retrieval


def get_ranked_chunk_ids(query, patient_id, search_type, k=5):
    """Run the chosen search and return top-k chunk_ids in rank order."""
    if search_type == "lexical":
        results = retrieval.search(
            query=query,
            patient_id=patient_id,
            num_results=k,
        )
    elif search_type == "semantic":
        results = retrieval.semantic_search(
            query=query,
            patient_id=patient_id,
            num_results=k,
        )
    elif search_type == "hybrid":
        results = retrieval.hybrid_search(
            query=query,
            patient_id=patient_id,
            num_results=k,
        )
    else:
        raise ValueError("search_type must be one of: lexical, semantic, hybrid")

    return [doc["chunk_id"] for doc in results]


def hit_rate_at_k(retrieved_ids, relevant_ids, k):
    """Hit@K: 1 if ANY relevant chunk appears in top K, else 0."""
    relevant = set(relevant_ids)
    top_k = retrieved_ids[:k]
    return int(any(doc_id in relevant for doc_id in top_k))


def mrr_at_k(retrieved_ids, relevant_ids, k):
    """MRR@K: 1/rank of first relevant chunk in top K, else 0."""
    relevant = set(relevant_ids)
    for rank, doc_id in enumerate(retrieved_ids[:k], start=1):
        if doc_id in relevant:
            return 1.0 / rank
    return 0.0

In [15]:
import math


def eval_search_type_df(gold_df, search_type, k=5):
    """
    Evaluate a given search_type over all rows in a gold DataFrame.

    gold_df columns:
      - patient_id
      - question_text  (or 'question' if you prefer)
      - gold_chunk_ids (list of chunk_ids)
    """
    hits = []
    mrrs = []

    for _, row in gold_df.iterrows():
        patient_id = row["patient_id"]
        question = row["question_text"]  # matches your patient_overview_gold schema
        gold_ids = row["gold_chunk_ids"]

        retrieved_ids = get_ranked_chunk_ids(
            query=question,
            patient_id=patient_id,
            search_type=search_type,
            k=k,
        )

        hits.append(hit_rate_at_k(retrieved_ids, gold_ids, k))
        mrrs.append(mrr_at_k(retrieved_ids, gold_ids, k))

    hit_rate = sum(hits) / len(hits) if hits else math.nan
    mrr = sum(mrrs) / len(mrrs) if mrrs else math.nan

    return hit_rate, mrr

In [16]:
# Filter just the patient_overview rows in case there are multiple question_types
po_gold = patient_overview_gold[
    patient_overview_gold["question_type"] == "patient_overview"
]

for st in ["lexical", "semantic", "hybrid"]:
    hr, mrr = eval_search_type_df(po_gold, st, k=5)
    print(f"[patient_overview] {st}: Hit@5 = {hr:.3f}, MRR@5 = {mrr:.3f}")

[patient_overview] lexical: Hit@5 = 0.778, MRR@5 = 0.189
[patient_overview] semantic: Hit@5 = 1.000, MRR@5 = 0.289
[patient_overview] hybrid: Hit@5 = 1.000, MRR@5 = 0.313


In [17]:
po_gold = conditions_gold[conditions_gold["question_type"] == "conditions"]

for st in ["lexical", "semantic", "hybrid"]:
    hr, mrr = eval_search_type_df(po_gold, st, k=5)
    print(f"[conditions_gold] {st}: Hit@5 = {hr:.3f}, MRR@5 = {mrr:.3f}")

[conditions_gold] lexical: Hit@5 = 0.111, MRR@5 = 0.037
[conditions_gold] semantic: Hit@5 = 1.000, MRR@5 = 0.422
[conditions_gold] hybrid: Hit@5 = 0.556, MRR@5 = 0.231


In [18]:
po_gold = medications_gold[medications_gold["question_type"] == "medications"]

for st in ["lexical", "semantic", "hybrid"]:
    hr, mrr = eval_search_type_df(po_gold, st, k=5)
    print(f"[medications_gold] {st}: Hit@5 = {hr:.3f}, MRR@5 = {mrr:.3f}")

[medications_gold] lexical: Hit@5 = 0.000, MRR@5 = 0.000
[medications_gold] semantic: Hit@5 = 1.000, MRR@5 = 0.741
[medications_gold] hybrid: Hit@5 = 0.333, MRR@5 = 0.081


In [19]:
# Filter just the oncology_timeline rows if you have multiple question_types
po_gold = oncology_timeline_gold[
    oncology_timeline_gold["question_type"] == "oncology_timeline"
]

for st in ["lexical", "semantic", "hybrid"]:
    hr, mrr = eval_search_type_df(po_gold, st, k=5)
    print(f"[oncology_timeline_gold] {st}: Hit@5 = {hr:.3f}, MRR@5 = {mrr:.3f}")

[oncology_timeline_gold] lexical: Hit@5 = 1.000, MRR@5 = 1.000
[oncology_timeline_gold] semantic: Hit@5 = 1.000, MRR@5 = 0.944
[oncology_timeline_gold] hybrid: Hit@5 = 1.000, MRR@5 = 1.000


## Evaluate retrieval with different boost values

Boost doesn't have much effect because fields don't compete


In [20]:
# helper function to wrap the search call with optional filters and boosts
import retrieval


def search(
    query,
    patient_id=None,
    doc_types=None,
    is_oncology=None,
    num_results=5,
    boost_dict=None,
):
    if boost_dict is None:
        boost_dict = {
            "title": 1.2,
            "heading": 2.0,
            "chunk_text": 1.0,
        }

    filter_dict = {}

    if patient_id is not None:
        filter_dict["patient_id"] = patient_id

    if doc_types is not None:
        filter_dict["doc_type"] = doc_types

    if is_oncology is not None:
        filter_dict["is_oncology"] = str(int(bool(is_oncology)))

    return retrieval.load_index().search(
        query=query,
        filter_dict=filter_dict,
        boost_dict=boost_dict,
        num_results=num_results,
    )

In [21]:
def get_ranked_chunk_ids(query, patient_id, search_type, k=5, boost_dict=None):
    """Run the chosen search and return top-k chunk_ids in rank order."""
    if search_type == "lexical":
        results = search(
            query=query,
            patient_id=patient_id,
            num_results=k,
            boost_dict=boost_dict,
        )
    elif search_type == "semantic":
        results = retrieval.semantic_search(
            query=query,
            patient_id=patient_id,
            num_results=k,
        )
    elif search_type == "hybrid":
        results = retrieval.hybrid_search(
            query=query,
            patient_id=patient_id,
            num_results=k,
        )
    else:
        raise ValueError("search_type must be one of: lexical, semantic, hybrid")

    return [doc["chunk_id"] for doc in results]

In [22]:
boost_configs = [
    {"name": "baseline", "boost": {"title": 1.2, "heading": 2.0, "chunk_text": 1.0}},
    {
        "name": "heading_strong",
        "boost": {"title": 1.0, "heading": 3.0, "chunk_text": 1.0},
    },
    {
        "name": "title_strong",
        "boost": {"title": 2.5, "heading": 2.0, "chunk_text": 1.0},
    },
    {
        "name": "chunk_strong",
        "boost": {"title": 1.0, "heading": 1.5, "chunk_text": 2.0},
    },
    {"name": "balanced", "boost": {"title": 1.5, "heading": 1.5, "chunk_text": 1.5}},
]

In [23]:
def eval_lexical_with_boost(gold_df, boost_dict, k=5):
    hits = []
    mrrs = []

    for _, row in gold_df.iterrows():
        patient_id = row["patient_id"]
        question = row["question_text"]
        gold_ids = row["gold_chunk_ids"]

        retrieved_ids = get_ranked_chunk_ids(
            query=question,
            patient_id=patient_id,
            search_type="lexical",
            k=k,
            boost_dict=boost_dict,
        )

        hits.append(hit_rate_at_k(retrieved_ids, gold_ids, k))
        mrrs.append(mrr_at_k(retrieved_ids, gold_ids, k))

    hit_rate = sum(hits) / len(hits) if hits else math.nan
    mrr = sum(mrrs) / len(mrrs) if mrrs else math.nan
    return hit_rate, mrr

In [24]:
po_gold = patient_overview_gold[
    patient_overview_gold["question_type"] == "patient_overview"
]

results_boost_search = []

for cfg in boost_configs:
    hr, mrr = eval_lexical_with_boost(po_gold, cfg["boost"], k=5)
    results_boost_search.append(
        {
            "config": cfg["name"],
            "title_boost": cfg["boost"]["title"],
            "heading_boost": cfg["boost"]["heading"],
            "chunk_text_boost": cfg["boost"]["chunk_text"],
            "hit_at_5": hr,
            "mrr_at_5": mrr,
        }
    )

import pandas as pd

boost_eval_df = pd.DataFrame(results_boost_search)
display(boost_eval_df)

,config,title_boost,heading_boost,chunk_text_boost,hit_at_5,mrr_at_5
0,baseline,1.2,2.0,1.0,0.777778,0.188889
1,heading_strong,1.0,3.0,1.0,0.777778,0.188889
2,title_strong,2.5,2.0,1.0,0.777778,0.188889
3,chunk_strong,1.0,1.5,2.0,0.777778,0.188889
4,balanced,1.5,1.5,1.5,0.777778,0.188889


In [25]:
cond_gold = conditions_gold[conditions_gold["question_type"] == "conditions"]

results_boost_search = []

for cfg in boost_configs:
    hr, mrr = eval_lexical_with_boost(cond_gold, cfg["boost"], k=5)
    results_boost_search.append(
        {
            "config": cfg["name"],
            "title_boost": cfg["boost"]["title"],
            "heading_boost": cfg["boost"]["heading"],
            "chunk_text_boost": cfg["boost"]["chunk_text"],
            "hit_at_5": hr,
            "mrr_at_5": mrr,
        }
    )

import pandas as pd

boost_eval_df = pd.DataFrame(results_boost_search)
display(boost_eval_df)

,config,title_boost,heading_boost,chunk_text_boost,hit_at_5,mrr_at_5
0,baseline,1.2,2.0,1.0,0.111111,0.037037
1,heading_strong,1.0,3.0,1.0,0.111111,0.037037
2,title_strong,2.5,2.0,1.0,0.111111,0.037037
3,chunk_strong,1.0,1.5,2.0,0.111111,0.037037
4,balanced,1.5,1.5,1.5,0.111111,0.037037


In [26]:
med_gold = medications_gold[medications_gold["question_type"] == "medications"]

results_boost_search = []

for cfg in boost_configs:
    hr, mrr = eval_lexical_with_boost(med_gold, cfg["boost"], k=5)
    results_boost_search.append(
        {
            "config": cfg["name"],
            "title_boost": cfg["boost"]["title"],
            "heading_boost": cfg["boost"]["heading"],
            "chunk_text_boost": cfg["boost"]["chunk_text"],
            "hit_at_5": hr,
            "mrr_at_5": mrr,
        }
    )

import pandas as pd

boost_eval_df = pd.DataFrame(results_boost_search)
display(boost_eval_df)

,config,title_boost,heading_boost,chunk_text_boost,hit_at_5,mrr_at_5
0,baseline,1.2,2.0,1.0,0.0,0.0
1,heading_strong,1.0,3.0,1.0,0.0,0.0
2,title_strong,2.5,2.0,1.0,0.0,0.0
3,chunk_strong,1.0,1.5,2.0,0.0,0.0
4,balanced,1.5,1.5,1.5,0.0,0.0


In [27]:
onc_gold = oncology_timeline_gold[
    oncology_timeline_gold["question_type"] == "oncology_timeline"
]

results_boost_search = []

for cfg in boost_configs:
    hr, mrr = eval_lexical_with_boost(onc_gold, cfg["boost"], k=5)
    results_boost_search.append(
        {
            "config": cfg["name"],
            "title_boost": cfg["boost"]["title"],
            "heading_boost": cfg["boost"]["heading"],
            "chunk_text_boost": cfg["boost"]["chunk_text"],
            "hit_at_5": hr,
            "mrr_at_5": mrr,
        }
    )

import pandas as pd

boost_eval_df = pd.DataFrame(results_boost_search)
display(boost_eval_df)

,config,title_boost,heading_boost,chunk_text_boost,hit_at_5,mrr_at_5
0,baseline,1.2,2.0,1.0,1.0,1.0
1,heading_strong,1.0,3.0,1.0,1.0,1.0
2,title_strong,2.5,2.0,1.0,1.0,1.0
3,chunk_strong,1.0,1.5,2.0,1.0,1.0
4,balanced,1.5,1.5,1.5,1.0,1.0


# Evaluate prompt

evaluate_relevance() is for “did the answer stay grounded in the retrieved context?”
evaluate_against_reference() is for “did the answer match the gold/reference summary?”

## Baseline RAG Pipeline

The following is the code in my first rag pipeline, which is now replaced by `retrieval.py` (data access) and `rag_service.py` (orchestration + prompting + LLM calls).


In [28]:
from pathlib import Path
from time import time
from functools import lru_cache
import json
import pickle

import numpy as np
from openai import OpenAI

from embedder import Embedder

client = OpenAI()

MODULE_DIR = Path.cwd().parent / "clinical_synopsis" / "unused"
REPO_ROOT = MODULE_DIR.parent.parent

INDEX_PATH = REPO_ROOT / "data" / "retrieval" / "minsearch_index.pkl"
VECTOR_INDEX_PATH = REPO_ROOT / "data" / "retrieval" / "vector_index.npz"
VECTOR_METADATA_PATH = REPO_ROOT / "data" / "retrieval" / "vector_index_metadata.json"

INSTRUCTIONS = """
Your task is to answer questions about a patient's clinical record
based only on the provided context.

Use the context to find relevant information and provide accurate answers.
If the answer is not found in the context, respond with "I don't know."

Do not make up facts that are not supported by the context.
When possible, mention the document type and date information that support the answer.
""".strip()

PROMPT_TEMPLATE = """
QUESTION: {question}

CONTEXT:
{context}
""".strip()

ENTRY_TEMPLATE = """
patient_id: {patient_id}
doc_type: {doc_type}
title: {title}
heading: {heading}
date_start: {date_start}
date_end: {date_end}
is_oncology: {is_oncology}
chunk_text: {chunk_text}
""".strip()

EVALUATION_PROMPT_TEMPLATE = """
You are an expert evaluator for a RAG system.

Your task is to evaluate the generated answer for:
1. Relevance to the user's question
2. Groundedness in the provided retrieved context

Classify relevance as one of:
- "NON_RELEVANT"
- "PARTLY_RELEVANT"
- "RELEVANT"

Classify groundedness as one of:
- "NOT_GROUNDED"
- "PARTLY_GROUNDED"
- "GROUNDED"

Question: {question}

Search type: {search_type}

Retrieved context:
{context}

Generated answer:
{answer}

Return parsable JSON only, without code fences, in exactly this format:

{{
  "Relevance": "NON_RELEVANT" | "PARTLY_RELEVANT" | "RELEVANT",
  "Groundedness": "NOT_GROUNDED" | "PARTLY_GROUNDED" | "GROUNDED",
  "Explanation": "[Provide a brief explanation for your evaluation]"
}}
""".strip()


def load_index(index_path=INDEX_PATH):
    with open(index_path, "rb") as f:
        return pickle.load(f)


@lru_cache(maxsize=1)
def load_vector_index():
    data = np.load(VECTOR_INDEX_PATH, allow_pickle=True)
    with open(VECTOR_METADATA_PATH, "r", encoding="utf-8") as f:
        metadata = json.load(f)

    embeddings = data["embeddings"]
    chunk_ids = data["chunk_ids"].tolist()
    documents = metadata["documents"]

    docs_by_chunk_id = {doc["chunk_id"]: doc for doc in documents}
    ordered_docs = [docs_by_chunk_id[chunk_id] for chunk_id in chunk_ids]

    return embeddings, ordered_docs


index = load_index()
vector_embeddings, vector_documents = load_vector_index()
embedder = Embedder()


def search(query, patient_id=None, doc_types=None, is_oncology=None, num_results=5):
    boost_dict = {
        "title": 1.0,
        "heading": 2.0,
        "chunk_text": 1.5,
    }

    filter_dict = {}

    if patient_id is not None:
        filter_dict["patient_id"] = patient_id

    if doc_types is not None:
        filter_dict["doc_type"] = doc_types

    if is_oncology is not None:
        filter_dict["is_oncology"] = str(int(bool(is_oncology)))

    return index.search(
        query=query,
        filter_dict=filter_dict,
        boost_dict=boost_dict,
        num_results=num_results,
    )


def semantic_search(
    query, patient_id=None, doc_types=None, is_oncology=None, num_results=5
):
    query_vector = embedder.encode(query, normalize=True)
    scores = vector_embeddings @ query_vector

    filtered = []
    for doc, score in zip(vector_documents, scores):
        if patient_id is not None and doc.get("patient_id") != patient_id:
            continue

        if doc_types is not None:
            if isinstance(doc_types, str):
                allowed_doc_types = {doc_types}
            else:
                allowed_doc_types = set(doc_types)

            if doc.get("doc_type") not in allowed_doc_types:
                continue

        if is_oncology is not None:
            if int(doc.get("is_oncology", 0)) != int(bool(is_oncology)):
                continue

        doc_with_score = dict(doc)
        doc_with_score["semantic_score"] = float(score)
        filtered.append(doc_with_score)

    filtered = sorted(filtered, key=lambda x: x["semantic_score"], reverse=True)
    return filtered[:num_results]


def rrf(result_lists, k=60, num_results=5):
    scores = {}
    docs = {}

    for results in result_lists:
        for rank, doc in enumerate(results, start=1):
            key = doc["chunk_id"]
            scores[key] = scores.get(key, 0.0) + 1 / (k + rank)
            docs[key] = doc

    ranked_keys = sorted(scores.keys(), key=lambda x: scores[x], reverse=True)

    fused = []
    for key in ranked_keys[:num_results]:
        doc = dict(docs[key])
        doc["rrf_score"] = scores[key]
        fused.append(doc)

    return fused


def hybrid_search(
    query, patient_id=None, doc_types=None, is_oncology=None, num_results=5, rrf_k=60
):
    lexical_results = search(
        query=query,
        patient_id=patient_id,
        doc_types=doc_types,
        is_oncology=is_oncology,
        num_results=10,
    )

    semantic_results = semantic_search(
        query=query,
        patient_id=patient_id,
        doc_types=doc_types,
        is_oncology=is_oncology,
        num_results=10,
    )

    return rrf([lexical_results, semantic_results], k=rrf_k, num_results=num_results)


def build_context(search_results):
    context = ""
    for doc in search_results:
        doc_copy = {
            "patient_id": doc.get("patient_id", ""),
            "doc_type": doc.get("doc_type", ""),
            "title": doc.get("title", ""),
            "heading": doc.get("heading", ""),
            "date_start": doc.get("date_start", ""),
            "date_end": doc.get("date_end", ""),
            "is_oncology": doc.get("is_oncology", ""),
            "chunk_text": doc.get("chunk_text", ""),
        }
        context = context + ENTRY_TEMPLATE.format(**doc_copy) + "\n\n"
    return context.strip()


def build_prompt(query, search_results):
    context = build_context(search_results)
    prompt = PROMPT_TEMPLATE.format(question=query, context=context).strip()
    return prompt


def calculate_openai_cost(model, tokens):
    pricing = {
        "gpt-5.4-mini": {
            "input_price_per_million": 0.75,
            "output_price_per_million": 4.50,
        },
    }

    info = pricing.get(model)
    if info is None:
        return {"input_cost": 0.0, "output_cost": 0.0, "total_cost": 0.0}

    input_tokens = tokens.get("input_tokens", 0)
    output_tokens = tokens.get("output_tokens", 0)

    input_cost = (input_tokens / 1_000_000) * info["input_price_per_million"]
    output_cost = (output_tokens / 1_000_000) * info["output_price_per_million"]
    total_cost = input_cost + output_cost

    return {
        "input_cost": input_cost,
        "output_cost": output_cost,
        "total_cost": total_cost,
    }


def llm(prompt, model="gpt-5.4-mini"):
    response = client.responses.create(
        model=model,
        input=[
            {
                "role": "developer",
                "content": [{"type": "input_text", "text": INSTRUCTIONS}],
            },
            {
                "role": "user",
                "content": [{"type": "input_text", "text": prompt}],
            },
        ],
    )

    answer = response.output_text.strip()

    usage = getattr(response, "usage", None)
    if usage is None:
        token_stats = {"input_tokens": 0, "output_tokens": 0, "total_tokens": 0}
    else:
        input_tokens = getattr(usage, "input_tokens", 0)
        output_tokens = getattr(usage, "output_tokens", 0)
        total_tokens = getattr(usage, "total_tokens", input_tokens + output_tokens)
        token_stats = {
            "input_tokens": input_tokens,
            "output_tokens": output_tokens,
            "total_tokens": total_tokens,
        }

    cost_info = calculate_openai_cost(model, token_stats)

    return {
        "answer": answer,
        "token_stats": token_stats,
        "cost": cost_info,
        "raw_response": response,
    }


def evaluate_relevance(question, answer, context, search_type, model="gpt-5.4-mini"):
    prompt = EVALUATION_PROMPT_TEMPLATE.format(
        question=question,
        answer=answer,
        context=context,
        search_type=search_type,
    )

    response = client.responses.create(
        model=model,
        input=[
            {
                "role": "developer",
                "content": [
                    {
                        "type": "input_text",
                        "text": "Return valid JSON only. Do not include markdown or code fences.",
                    }
                ],
            },
            {
                "role": "user",
                "content": [{"type": "input_text", "text": prompt}],
            },
        ],
    )

    evaluation_text = response.output_text.strip()

    usage = getattr(response, "usage", None)
    if usage is None:
        token_stats = {"input_tokens": 0, "output_tokens": 0, "total_tokens": 0}
    else:
        input_tokens = getattr(usage, "input_tokens", 0)
        output_tokens = getattr(usage, "output_tokens", 0)
        total_tokens = getattr(usage, "total_tokens", input_tokens + output_tokens)
        token_stats = {
            "input_tokens": input_tokens,
            "output_tokens": output_tokens,
            "total_tokens": total_tokens,
        }

    cost_info = calculate_openai_cost(model, token_stats)

    try:
        evaluation = json.loads(evaluation_text)
    except json.JSONDecodeError:
        evaluation = {
            "Relevance": "UNKNOWN",
            "Groundedness": "UNKNOWN",
            "Explanation": "Failed to parse evaluation",
        }

    return {
        "evaluation": evaluation,
        "token_stats": token_stats,
        "cost": cost_info,
        "raw_text": evaluation_text,
    }


def rag(
    query,
    patient_id=None,
    doc_types=None,
    is_oncology=None,
    num_results=5,
    model="gpt-5.4-mini",
    search_type="lexical",
):
    t0 = time()

    if search_type == "lexical":
        search_results = search(
            query=query,
            patient_id=patient_id,
            doc_types=doc_types,
            is_oncology=is_oncology,
            num_results=num_results,
        )
    elif search_type == "semantic":
        search_results = semantic_search(
            query=query,
            patient_id=patient_id,
            doc_types=doc_types,
            is_oncology=is_oncology,
            num_results=num_results,
        )
    elif search_type == "hybrid":
        search_results = hybrid_search(
            query=query,
            patient_id=patient_id,
            doc_types=doc_types,
            is_oncology=is_oncology,
            num_results=num_results,
        )
    else:
        raise ValueError("search_type must be one of: lexical, semantic, hybrid")

    context = build_context(search_results)
    prompt = build_prompt(query, search_results)

    llm_result = llm(prompt, model=model)
    answer = llm_result["answer"]
    token_stats = llm_result["token_stats"]
    answer_cost = llm_result["cost"]

    eval_result = evaluate_relevance(
        question=query,
        answer=answer,
        context=context,
        search_type=search_type,
        model=model,
    )
    evaluation = eval_result["evaluation"]
    eval_token_stats = eval_result["token_stats"]
    eval_cost = eval_result["cost"]

    took = time() - t0

    answer_data = {
        "answer": answer,
        "model_used": model,
        "search_type": search_type,
        "response_time": took,
        "relevance": evaluation.get("Relevance", "UNKNOWN"),
        "groundedness": evaluation.get("Groundedness", "UNKNOWN"),
        "evaluation_explanation": evaluation.get(
            "Explanation", "Failed to parse evaluation"
        ),
        "search_results": search_results,
        "prompt_tokens": token_stats["input_tokens"],
        "completion_tokens": token_stats["output_tokens"],
        "total_tokens": token_stats["total_tokens"],
        "answer_input_cost_usd": answer_cost["input_cost"],
        "answer_output_cost_usd": answer_cost["output_cost"],
        "answer_total_cost_usd": answer_cost["total_cost"],
        "eval_prompt_tokens": eval_token_stats["input_tokens"],
        "eval_completion_tokens": eval_token_stats["output_tokens"],
        "eval_total_tokens": eval_token_stats["total_tokens"],
        "eval_input_cost_usd": eval_cost["input_cost"],
        "eval_output_cost_usd": eval_cost["output_cost"],
        "eval_total_cost_usd": eval_cost["total_cost"],
        "overall_total_cost_usd": answer_cost["total_cost"] + eval_cost["total_cost"],
    }

    return answer_data

### LLM-as-judge evaluating against a reference summary

We define a reference-based judge (`evaluate_against_reference`) which, in addition to question and answer (scoring relevance and faithfulness), evaluates **gold-based reference** (scoring coverage).  

So while the answers are generated using only the retrieved context (the top-5 chunks that `rag` got from lexical/semantic/hybrid search, built into context by `build_context(search_results)`), now the reference-based judge sees the **full reference** built from **all gold_chunk_ids** with `build_reference_summary`, which is richer than just the top-5 context. This means that although Hit@5 and MRR@5 metrics might be great, i.e., the right chunks get to the top, the answers may be inaccurate or incomplete relative to the gold truth.

Strikingly, the reference-judge concludes that the answers introduce some hallucinated information that is not present in the reference summary. 

### Overview question 
"Give a concise overview of this patient’s medical background and current care context."

For the 9 ground truth patients with hybrid search (see `po_ref_eval_hybrid` table below), the relevance is rated as PARTLY_RELEVANT five times, UKNOWN two times, but and faithfulness is both PARTLY_FAITHFUL (four times) and NOT_FAITHFUL (five times). Coverage is once INSUFFICIENT", the rest is PARTIAL. 
This suggests that the answers are poorly grounded in the provided reference material.

With lexical search (`po_ref_eval_lexical` below) coverage improves slightly: it is once COMPREHENSIVE (while PARTLY_FAITHFUL), rest is PARTIAL. Faithfulness is better than with hybrid search: two times NOT_FAITHFUL, seven times PARTLY_FAITHFUL.

Hybrid search being worse on faithfulness suggests that hybrid retrieval sometimes brings extra chunks, increasing the chance of inaccuracies.


### Conditions question
"What are the patient’s main diagnosed conditions?"

The reference-based judge claims that the answers are not just missing some gold conditions (which we expect because the reference is richer than the context), they’re also inventing conditions that are not in the data and misinterpreting the status (treating resolved as active, or vice versa).

Manually inspecting the high complexity patients I see that both the answers and the reference-based judge are wrong, because they have trouble putting together the information in the chunks from patient_overview.md and conditions.csv.

This is because patient_overview.md contains top 10 diagnostic reports, as an approximation for "recent results", which for complex patients with lots of records is an incomplete source for the summary. Reaching directly to the full record, conditions.csv, the two LLMs selects the most serious diagnoses from the life-time record and then are not able to infer which are the recent and/or active diagnoses.


### Conclusion
I conclude that the poor accuracy of the answers is due to poor generation not retrieval. With a complex source (Recent Conditions heading in patient_overview.md + conditions.csv), the answer quality depends heavily on the **prompt and context**, not just retrieval.

In [29]:
all_gold_df = pd.concat(
    [
        patient_overview_gold,
        conditions_gold,
        medications_gold,
        oncology_timeline_gold,
    ],
    ignore_index=True,
)

all_gold_df

,patient_id,question_type,question_text,gold_chunk_ids
0,29f6beee-162f-0113-7884-72245814693f,patient_overview,Give a concise overview of this patient’s medi...,"[abc678e3f0fdb2313c03b92ff62bf06f08120f7b, c1b..."
1,3a1c7c7b-0f87-e7ba-f2ed-1b0882fe3678,patient_overview,Give a concise overview of this patient’s medi...,"[a970cc1ae6abca57a8813b4cc2aea2df1d41cde1, 519..."
2,41681ed6-efc5-94c0-1bc0-f60b34dbd31b,patient_overview,Give a concise overview of this patient’s medi...,"[0590211551c10b6259f5a9062003667e93e8f4e6, 9c7..."
3,4736727e-63f4-071a-1516-a49310f5a052,patient_overview,Give a concise overview of this patient’s medi...,"[63f01e07f0f326ce090ed1d0e1e9cb9ddb88d53b, 49c..."
4,aee216e6-cbe8-eaf2-3241-4bd1e8a01494,patient_overview,Give a concise overview of this patient’s medi...,"[aff56cb2f4bb43dff398ad5e744d1f5a364b7670, 280..."
5,d65197b3-056a-2136-b584-77f43c29da3f,patient_overview,Give a concise overview of this patient’s medi...,"[abc3ffd40cbb13d210ac910a0f5aaa43b998a587, e84..."
6,ecc4a7d0-8838-36b4-44ba-676d5a1f7927,patient_overview,Give a concise overview of this patient’s medi...,"[aee14674ce11ab5daa05de7c09f81db1f25365a4, 95b..."
7,f203e11d-5573-1624-69b8-af8436987b3e,patient_overview,Give a concise overview of this patient’s medi...,"[1f9771cf6cdb8353d65819a133ea34b11b3b8e58, 78c..."
8,f3739580-797d-ae04-eebf-aeddb2fc2f64,patient_overview,Give a concise overview of this patient’s medi...,"[a9165d0158523ea3b0c882ba42cd1f37f02f067d, c59..."
9,29f6beee-162f-0113-7884-72245814693f,conditions,What are the patient’s main diagnosed conditions?,"[14326c002878a932e87427ea2a9ba2b6e6ad1404, abc..."


In [30]:
def build_reference_summary(chunks_df, gold_chunk_ids):
    rows = chunks_df.loc[chunks_df["chunk_id"].isin(gold_chunk_ids)]
    texts = rows["chunk_text"].astype(str).tolist()
    return "\n\n".join(texts)

In [31]:
REFERENCE_EVAL_PROMPT_TEMPLATE = """
You are an expert evaluator for a clinical RAG system.

Your task is to evaluate the generated answer against a reference summary
for three criteria:

1. Relevance to the user's question
2. Faithfulness to the reference (no contradictions or hallucinations)
3. Coverage of the key points in the reference that are relevant to the question

Classify relevance as one of:
- "NON_RELEVANT"
- "PARTLY_RELEVANT"
- "RELEVANT"

Classify faithfulness as one of:
- "NOT_FAITHFUL"
- "PARTLY_FAITHFUL"
- "FAITHFUL"

Classify coverage as one of:
- "INSUFFICIENT"
- "PARTIAL"
- "COMPREHENSIVE"

Question: {question}

Generated answer:
{answer}

Reference summary:
{reference}

Return parsable JSON only, without code fences, in exactly this format:

{{
  "Relevance": "NON_RELEVANT" | "PARTLY_RELEVANT" | "RELEVANT",
  "Faithfulness": "NOT_FAITHFUL" | "PARTLY_FAITHFUL" | "FAITHFUL",
  "Coverage": "INSUFFICIENT" | "PARTIAL" | "COMPREHENSIVE",
  "Explanation": "[Provide a brief explanation for your evaluation]"
}}
""".strip()

In [32]:
def evaluate_against_reference(question, answer, reference, model="gpt-5.4-mini"):
    """LLM-as-judge for relevance, faithfulness, and coverage vs a reference summary."""
    prompt = REFERENCE_EVAL_PROMPT_TEMPLATE.format(
        question=question,
        answer=answer,
        reference=reference,
    )

    response = client.responses.create(
        model=model,
        input=[
            {
                "role": "developer",
                "content": [
                    {
                        "type": "input_text",
                        "text": "Return valid JSON only. Do not include markdown or code fences.",
                    }
                ],
            },
            {
                "role": "user",
                "content": [
                    {
                        "type": "input_text",
                        "text": prompt,
                    }
                ],
            },
        ],
    )

    evaluation_text = response.output_text.strip()

    usage = getattr(response, "usage", None)
    if usage is None:
        token_stats = {
            "input_tokens": 0,
            "output_tokens": 0,
            "total_tokens": 0,
        }
    else:
        input_tokens = getattr(usage, "input_tokens", 0)
        output_tokens = getattr(usage, "output_tokens", 0)
        total_tokens = getattr(usage, "total_tokens", input_tokens + output_tokens)
        token_stats = {
            "input_tokens": input_tokens,
            "output_tokens": output_tokens,
            "total_tokens": total_tokens,
        }

    cost_info = calculate_openai_cost(model, token_stats)

    try:
        evaluation = json.loads(evaluation_text)
    except json.JSONDecodeError:
        evaluation = {
            "Relevance": "UNKNOWN",
            "Faithfulness": "UNKNOWN",
            "Coverage": "UNKNOWN",
            "Explanation": "Failed to parse evaluation",
        }

    return {
        "evaluation": evaluation,
        "token_stats": token_stats,
        "cost": cost_info,
        "raw_text": evaluation_text,
    }

In [33]:
def run_reference_eval(
    gold_df,
    chunks_df,
    search_type="hybrid",
    model="gpt-5.4-mini",
    num_results=5,
):
    eval_rows = []

    for _, row in gold_df.iterrows():
        patient_id = row["patient_id"]
        question_type = row["question_type"]
        question = row["question_text"]
        gold_ids = row["gold_chunk_ids"]

        # 1. Build reference summary from gold chunks
        reference = build_reference_summary(chunks_df, gold_ids)

        # 2. Run the notebook RAG to get an answer
        rag_out = rag(
            query=question,
            patient_id=patient_id,
            doc_types=None,  # or restrict per question_type if you want
            is_oncology=None,
            num_results=num_results,
            model=model,
            search_type=search_type,
        )

        answer = rag_out["answer"]

        # 3. Judge answer vs reference (using notebook judge)
        judge_out = evaluate_against_reference(
            question=question,
            answer=answer,
            reference=reference,
            model=model,
        )

        evaluation = judge_out["evaluation"]

        eval_rows.append(
            {
                "patient_id": patient_id,
                "question_type": question_type,
                "search_type": search_type,
                "question_text": question,
                "gold_chunk_ids": gold_ids,
                "reference_summary": reference,
                "model_answer": answer,
                # Judge scores
                "judge_relevance": evaluation.get("Relevance", "UNKNOWN"),
                "judge_faithfulness": evaluation.get("Faithfulness", "UNKNOWN"),
                "judge_coverage": evaluation.get("Coverage", "UNKNOWN"),
                "judge_explanation": evaluation.get("Explanation", ""),
                # Answer token / cost (from rag.rag)
                "prompt_tokens": rag_out.get("prompt_tokens"),
                "completion_tokens": rag_out.get("completion_tokens"),
                "total_tokens": rag_out.get("total_tokens"),
                "answer_input_cost_usd": rag_out.get("answer_input_cost_usd"),
                "answer_output_cost_usd": rag_out.get("answer_output_cost_usd"),
                "answer_total_cost_usd": rag_out.get("answer_total_cost_usd"),
                # Eval token / cost (from rag.rag)
                "eval_prompt_tokens": rag_out.get("eval_prompt_tokens"),
                "eval_completion_tokens": rag_out.get("eval_completion_tokens"),
                "eval_total_tokens": rag_out.get("eval_total_tokens"),
                "eval_input_cost_usd": rag_out.get("eval_input_cost_usd"),
                "eval_output_cost_usd": rag_out.get("eval_output_cost_usd"),
                "eval_total_cost_usd": rag_out.get("eval_total_cost_usd"),
                "overall_total_cost_usd": rag_out.get("overall_total_cost_usd"),
            }
        )

    return pd.DataFrame(eval_rows)

In [34]:
# Evaluate patient overview questions with hybrid search
po_gold = patient_overview_gold[
    patient_overview_gold["question_type"] == "patient_overview"
]

po_ref_eval_hybrid = run_reference_eval(
    gold_df=po_gold,
    chunks_df=chunks_df,
    search_type="hybrid",
    model="gpt-5.4-mini",
    num_results=5,
)

po_ref_eval_hybrid

,patient_id,question_type,search_type,question_text,gold_chunk_ids,reference_summary,model_answer,judge_relevance,judge_faithfulness,judge_coverage,...,answer_input_cost_usd,answer_output_cost_usd,answer_total_cost_usd,eval_prompt_tokens,eval_completion_tokens,eval_total_tokens,eval_input_cost_usd,eval_output_cost_usd,eval_total_cost_usd,overall_total_cost_usd
0,29f6beee-162f-0113-7884-72245814693f,patient_overview,hybrid,Give a concise overview of this patient’s medi...,"[abc678e3f0fdb2313c03b92ff62bf06f08120f7b, c1b...",- Total score [DAST-10]; value: 2.0 {score}; d...,This is an oncology patient: a female born 198...,PARTLY_RELEVANT,PARTLY_FAITHFUL,PARTIAL,...,0.001292,0.000963,0.002255,2064,84,2148,0.001548,0.000378,0.001926,0.004181
1,3a1c7c7b-0f87-e7ba-f2ed-1b0882fe3678,patient_overview,hybrid,Give a concise overview of this patient’s medi...,"[a970cc1ae6abca57a8813b4cc2aea2df1d41cde1, 519...",- Total score [AUDIT-C]; value: 1.0 {score}; d...,This is a female patient born 1969-05-12. Her ...,RELEVANT,PARTLY_FAITHFUL,PARTIAL,...,0.001335,0.000878,0.002213,2102,88,2190,0.001577,0.000396,0.001973,0.004185
2,41681ed6-efc5-94c0-1bc0-f60b34dbd31b,patient_overview,hybrid,Give a concise overview of this patient’s medi...,"[0590211551c10b6259f5a9062003667e93e8f4e6, 9c7...",- Cancer Disease Progression; value: Patient's...,This is a female patient born 1974-04-10. The ...,PARTLY_RELEVANT,PARTLY_FAITHFUL,PARTIAL,...,0.001555,0.001435,0.002990,2519,96,2615,0.001889,0.000432,0.002321,0.005312
3,4736727e-63f4-071a-1516-a49310f5a052,patient_overview,hybrid,Give a concise overview of this patient’s medi...,"[63f01e07f0f326ce090ed1d0e1e9cb9ddb88d53b, 49c...",- Cancer Disease Progression; value: Patient's...,This is an oncology patient overview record fo...,RELEVANT,PARTLY_FAITHFUL,PARTIAL,...,0.001269,0.001287,0.002556,2105,139,2244,0.001579,0.000625,0.002204,0.004760
4,aee216e6-cbe8-eaf2-3241-4bd1e8a01494,patient_overview,hybrid,Give a concise overview of this patient’s medi...,"[aff56cb2f4bb43dff398ad5e744d1f5a364b7670, 280...",- Total score [DAST-10]; value: 1.0 {score}; d...,This is a female patient born **1984-06-18** w...,PARTLY_RELEVANT,PARTLY_FAITHFUL,PARTIAL,...,0.001567,0.001215,0.002782,2487,129,2616,0.001865,0.000580,0.002446,0.005228
5,d65197b3-056a-2136-b584-77f43c29da3f,patient_overview,hybrid,Give a concise overview of this patient’s medi...,"[abc3ffd40cbb13d210ac910a0f5aaa43b998a587, e84...",- Cancer Disease Progression; value: Patient's...,This is a female pediatric oncology patient (b...,RELEVANT,PARTLY_FAITHFUL,PARTIAL,...,0.001237,0.000770,0.002007,1948,117,2065,0.001461,0.000526,0.001988,0.003994
6,ecc4a7d0-8838-36b4-44ba-676d5a1f7927,patient_overview,hybrid,Give a concise overview of this patient’s medi...,"[aee14674ce11ab5daa05de7c09f81db1f25365a4, 95b...",- Cancer Disease Progression; value: Patient's...,This is a female patient born 1947-04-19. The ...,RELEVANT,PARTLY_FAITHFUL,PARTIAL,...,0.001546,0.000783,0.002329,2362,116,2478,0.001772,0.000522,0.002293,0.004622
7,f203e11d-5573-1624-69b8-af8436987b3e,patient_overview,hybrid,Give a concise overview of this patient’s medi...,"[1f9771cf6cdb8353d65819a133ea34b11b3b8e58, 78c...",- Total score [AUDIT-C]; value: 1.0 {score}; d...,This is a female patient born on 1965-01-05. H...,PARTLY_RELEVANT,PARTLY_FAITHFUL,PARTIAL,...,0.001492,0.000837,0.002329,2302,101,2403,0.001726,0.000454,0.002181,0.004510
8,f3739580-797d-ae04-eebf-aeddb2fc2f64,patient_overview,hybrid,Give a concise overview of this patient’s medi...,"[a9165d0158523ea3b0c882ba42cd1f37f02f067d, c59...",- Cancer Disease Progression; value: Patient's...,"This is a female oncology patient, born 2019-0...",RELEVANT,PARTLY_FAITHFUL,PARTIAL,...,0.001562,0.000680,0.002242,2361,81,2442,0.001771,0.000365,0.002135,0.004377


In [35]:
# Evaluate patient overview questions with lexical search
po_gold = patient_overview_gold[
    patient_overview_gold["question_type"] == "patient_overview"
]

po_ref_eval_lexical = run_reference_eval(
    gold_df=po_gold,
    chunks_df=chunks_df,
    search_type="lexical",
    model="gpt-5.4-mini",
    num_results=5,
)

po_ref_eval_lexical
# Coverage is once "COMPREHENSIVE" (while "PARTLY_FAITHFUL"), rest "PARTIAL".

,patient_id,question_type,search_type,question_text,gold_chunk_ids,reference_summary,model_answer,judge_relevance,judge_faithfulness,judge_coverage,...,answer_input_cost_usd,answer_output_cost_usd,answer_total_cost_usd,eval_prompt_tokens,eval_completion_tokens,eval_total_tokens,eval_input_cost_usd,eval_output_cost_usd,eval_total_cost_usd,overall_total_cost_usd
0,29f6beee-162f-0113-7884-72245814693f,patient_overview,lexical,Give a concise overview of this patient’s medi...,"[abc678e3f0fdb2313c03b92ff62bf06f08120f7b, c1b...",- Total score [DAST-10]; value: 2.0 {score}; d...,This is a female patient born 1980-05-15. The ...,PARTLY_RELEVANT,NOT_FAITHFUL,PARTIAL,...,0.001288,0.000976,0.002265,2062,82,2144,0.001547,0.000369,0.001916,0.004181
1,3a1c7c7b-0f87-e7ba-f2ed-1b0882fe3678,patient_overview,lexical,Give a concise overview of this patient’s medi...,"[a970cc1ae6abca57a8813b4cc2aea2df1d41cde1, 519...",- Total score [AUDIT-C]; value: 1.0 {score}; d...,This is a female patient born on 1969-05-12. B...,PARTLY_RELEVANT,PARTLY_FAITHFUL,PARTIAL,...,0.001436,0.000864,0.002300,2234,79,2313,0.001675,0.000355,0.002031,0.004331
2,41681ed6-efc5-94c0-1bc0-f60b34dbd31b,patient_overview,lexical,Give a concise overview of this patient’s medi...,"[0590211551c10b6259f5a9062003667e93e8f4e6, 9c7...",- Cancer Disease Progression; value: Patient's...,"This is a female patient born 1974-04-10, with...",RELEVANT,PARTLY_FAITHFUL,PARTIAL,...,0.001363,0.000810,0.002173,2124,75,2199,0.001593,0.000337,0.001930,0.004103
3,4736727e-63f4-071a-1516-a49310f5a052,patient_overview,lexical,Give a concise overview of this patient’s medi...,"[63f01e07f0f326ce090ed1d0e1e9cb9ddb88d53b, 49c...",- Cancer Disease Progression; value: Patient's...,This is a female patient born on 2018-01-07. T...,PARTLY_RELEVANT,NOT_FAITHFUL,INSUFFICIENT,...,0.001170,0.000792,0.001962,1863,103,1966,0.001397,0.000463,0.001861,0.003823
4,aee216e6-cbe8-eaf2-3241-4bd1e8a01494,patient_overview,lexical,Give a concise overview of this patient’s medi...,"[aff56cb2f4bb43dff398ad5e744d1f5a364b7670, 280...",- Total score [DAST-10]; value: 1.0 {score}; d...,This is a female patient born 1984-06-18. The ...,PARTLY_RELEVANT,PARTLY_FAITHFUL,PARTIAL,...,0.001375,0.001148,0.002523,2216,93,2309,0.001662,0.000418,0.002080,0.004603
5,d65197b3-056a-2136-b584-77f43c29da3f,patient_overview,lexical,Give a concise overview of this patient’s medi...,"[abc3ffd40cbb13d210ac910a0f5aaa43b998a587, e84...",- Cancer Disease Progression; value: Patient's...,This is a young female patient with an active ...,RELEVANT,PARTLY_FAITHFUL,PARTIAL,...,0.001029,0.000959,0.001988,1712,85,1797,0.001284,0.000383,0.001667,0.003654
6,ecc4a7d0-8838-36b4-44ba-676d5a1f7927,patient_overview,lexical,Give a concise overview of this patient’s medi...,"[aee14674ce11ab5daa05de7c09f81db1f25365a4, 95b...",- Cancer Disease Progression; value: Patient's...,This is a female patient born 1947-04-19 with ...,RELEVANT,PARTLY_FAITHFUL,PARTIAL,...,0.001270,0.000580,0.001850,1949,107,2056,0.001462,0.000481,0.001943,0.003793
7,f203e11d-5573-1624-69b8-af8436987b3e,patient_overview,lexical,Give a concise overview of this patient’s medi...,"[1f9771cf6cdb8353d65819a133ea34b11b3b8e58, 78c...",- Total score [AUDIT-C]; value: 1.0 {score}; d...,This patient is a female born on 1965-01-05. T...,RELEVANT,PARTLY_FAITHFUL,PARTIAL,...,0.001260,0.000882,0.002142,2003,85,2088,0.001502,0.000383,0.001885,0.004027
8,f3739580-797d-ae04-eebf-aeddb2fc2f64,patient_overview,lexical,Give a concise overview of this patient’s medi...,"[a9165d0158523ea3b0c882ba42cd1f37f02f067d, c59...",- Cancer Disease Progression; value: Patient's...,This is an oncology patient: a female born in ...,RELEVANT,PARTLY_FAITHFUL,COMPREHENSIVE,...,0.001132,0.001184,0.002315,1899,99,1998,0.001424,0.000445,0.001870,0.004185


In [36]:
# Evaluate conditions question with hybrid search
cond_gold = conditions_gold[conditions_gold["question_type"] == "conditions"]

cond_ref_eval_hybrid = run_reference_eval(
    gold_df=cond_gold,
    chunks_df=chunks_df,
    search_type="hybrid",
    model="gpt-5.4-mini",
    num_results=5,
)

cond_ref_eval_hybrid

,patient_id,question_type,search_type,question_text,gold_chunk_ids,reference_summary,model_answer,judge_relevance,judge_faithfulness,judge_coverage,...,answer_input_cost_usd,answer_output_cost_usd,answer_total_cost_usd,eval_prompt_tokens,eval_completion_tokens,eval_total_tokens,eval_input_cost_usd,eval_output_cost_usd,eval_total_cost_usd,overall_total_cost_usd
0,29f6beee-162f-0113-7884-72245814693f,conditions,hybrid,What are the patient’s main diagnosed conditions?,"[14326c002878a932e87427ea2a9ba2b6e6ad1404, abc...",- Not in labor force (finding); date: 2022-06-...,The main diagnosed conditions shown in the **c...,RELEVANT,PARTLY_FAITHFUL,PARTIAL,...,0.001327,0.000540,0.001867,2016,76,2092,0.001512,0.000342,0.001854,0.003721
1,3a1c7c7b-0f87-e7ba-f2ed-1b0882fe3678,conditions,hybrid,What are the patient’s main diagnosed conditions?,"[772ce44ecf1664ce3cd0f321962e31b1afeaf658, a97...",- Victim of intimate partner abuse (finding); ...,The main diagnosed condition visible in the pr...,PARTLY_RELEVANT,NOT_FAITHFUL,INSUFFICIENT,...,0.001401,0.000522,0.001923,2111,105,2216,0.001583,0.000473,0.002056,0.003979
2,41681ed6-efc5-94c0-1bc0-f60b34dbd31b,conditions,hybrid,What are the patient’s main diagnosed conditions?,"[8123905ce9ef5c2a100a0400f86029469183dbb2, 059...",- Full-time employment (finding); date: 2022-0...,The patient’s main diagnosed conditions in the...,RELEVANT,PARTLY_FAITHFUL,PARTIAL,...,0.001525,0.000486,0.002011,2268,71,2339,0.001701,0.000320,0.002021,0.004031
3,4736727e-63f4-071a-1516-a49310f5a052,conditions,hybrid,What are the patient’s main diagnosed conditions?,"[7ac2bda6270220cb79ace629a38ef8b5a13ddf31, 63f...",- Facial laceration; date: 2022-01-30T15:16:05...,"The patient’s main diagnosed conditions, based...",PARTLY_RELEVANT,PARTLY_FAITHFUL,PARTIAL,...,0.001040,0.000531,0.001571,1632,101,1733,0.001224,0.000454,0.001679,0.003250
4,aee216e6-cbe8-eaf2-3241-4bd1e8a01494,conditions,hybrid,What are the patient’s main diagnosed conditions?,"[e733276d7db1b542f556b25839e3602be1339cbc, aff...",- Sprain of wrist; date: 2022-05-31T18:08:50-0...,The main diagnosed conditions shown in the rec...,RELEVANT,PARTLY_FAITHFUL,PARTIAL,...,0.001313,0.000544,0.001857,1998,85,2083,0.001499,0.000383,0.001881,0.003738
5,d65197b3-056a-2136-b584-77f43c29da3f,conditions,hybrid,What are the patient’s main diagnosed conditions?,"[45b0f62c2348b66d770620e4e5acfe630cf41f96, abc...",- Otitis media; date: 2021-06-16T04:22:18-04:0...,The patient’s main diagnosed conditions are:\n...,RELEVANT,FAITHFUL,PARTIAL,...,0.001388,0.000454,0.001842,2079,67,2146,0.001559,0.000302,0.001861,0.003703
6,ecc4a7d0-8838-36b4-44ba-676d5a1f7927,conditions,hybrid,What are the patient’s main diagnosed conditions?,"[05d547f22443d60a0d31fa1776bed869da7459cd, aee...",- Malignant neoplasm of breast (disorder); dat...,The main diagnosed conditions shown in the rec...,PARTLY_RELEVANT,NOT_FAITHFUL,INSUFFICIENT,...,0.001228,0.000616,0.001844,1901,94,1995,0.001426,0.000423,0.001849,0.003693
7,f203e11d-5573-1624-69b8-af8436987b3e,conditions,hybrid,What are the patient’s main diagnosed conditions?,"[f73c3623d5df217075882e25106a9d6388dde40e, 1f9...",- Concussion with loss of consciousness; date:...,The main diagnosed conditions in the record in...,RELEVANT,PARTLY_FAITHFUL,PARTIAL,...,0.001252,0.001278,0.002530,2080,66,2146,0.001560,0.000297,0.001857,0.004387
8,f3739580-797d-ae04-eebf-aeddb2fc2f64,conditions,hybrid,What are the patient’s main diagnosed conditions?,"[4c17e4784d257f85e9fd1ddd74054e5dc3adae9f, a91...",- Otitis media; date: 2022-03-27T21:25:37-04:0...,"The patient’s main diagnosed conditions, based...",RELEVANT,FAITHFUL,COMPREHENSIVE,...,0.001162,0.000571,0.001734,1804,123,1927,0.001353,0.000554,0.001907,0.003641


## Question-Routed RAG Pipeline


In the section above we see the problem: when the question is broad (“main diagnosed conditions”) and the prompt allows the model to treat the task as “interpret and summarize like a clinician”, not “extract from a structured list.” We want this general behavior for overview questions, but not condition/medication/oncology questions.

In the final version of the Clinical Synopsis app, I implement **task-specific prompting** and **stricter context selection**, by adding question-type–specific instructions to the prompt build.

A **question router** selects among 4 question types (Patient overview, Conditions, Medications, Oncology Timeline) and controls the prompt and retrieval (which now covers selected category-specific document types and document headings).

We now have BASE_INSTRUCTIONS: general behavior (be concise, don’t hallucinate, etc.), and EXTRA instructions for question-type specific behavior. (I spent a whole day re-formulating the EXTRA instructions to achive acceptable outputs. I tested too many prompts to include them here; some version worked on some patients but failed on others, etc.)

E.g., for conditions questions the router selects `prompt_mode="extract_conditions"`, which uses CONDITIONS_EXTRA to do structured condition extraction. The answer now correctly lists all the active diagnoses and separates medical diagnoses from socio-economic findings.

**Router implementation in the app**
The user types free text. Because we have only 4 supported question types, the user can pick one, but doesn't have to. If the user doesn't choose the question type, the router guesses. If it's confidence is low, it forces the user to choose.

E.g. For the query "Summarize this patient's oncology history", the router responds: "Suggested question type: Oncology timeline (routing confidence: 60%)."

E.g. for the query "What happened after treatment?", the router says "I could not confidently classify this question. Please choose the question type."

In [37]:
from question_router import route_question

test_questions = [
    "Provide a brief overview of this patient's medical background and current status.",
    "What are this patient's main diagnosed conditions and their status?",
    "What medications is this patient currently or recently taking?",
    "Summarize this patient's oncology history.",
    "What happened after treatment?",
    "Any evidence of progression?",
]

for question in test_questions:
    route = route_question(question)
    print("Q:", question)
    print("Route:", route.question_type)
    print("Confidence:", round(route.confidence, 2))
    print("Scores:", route.scores)
    print("-" * 60)

Q: Provide a brief overview of this patient's medical background and current status.
Route: patient_overview
Confidence: 1.0
Scores: {'patient_overview': 6.0, 'conditions': 0.0, 'medications': 0.0, 'oncology_timeline': 0.0}
------------------------------------------------------------
Q: What are this patient's main diagnosed conditions and their status?
Route: conditions
Confidence: 1.0
Scores: {'patient_overview': 0.5, 'conditions': 7.0, 'medications': 0.0, 'oncology_timeline': 0.0}
------------------------------------------------------------
Q: What medications is this patient currently or recently taking?
Route: medications
Confidence: 1.0
Scores: {'patient_overview': 0.0, 'conditions': 0.0, 'medications': 6.0, 'oncology_timeline': 0.0}
------------------------------------------------------------
Q: Summarize this patient's oncology history.
Route: oncology_timeline
Confidence: 0.6
Scores: {'patient_overview': 1.0, 'conditions': 0.0, 'medications': 0.0, 'oncology_timeline': 3.0}
---

### Example patient with question routing

The answer correctly lists all the active diagnoses and separates medical diagnoses from socio-economic findings.

The judge's result is the same as without routing (relevance_score is 2, groundedness_score: 1, overall_score: 1.5, RELEVANT, PARTLY_GROUNDED), but without routing the cancer diagnosis is missed, see next section.

In [38]:
import rag_service as rag

In [39]:
patient_id = "f203e11d-5573-1624-69b8-af8436987b3e"  # high complexity patient

search_type = "hybrid"
model = "gpt-5.4-mini"
num_results = 5

In [40]:
# Helper functions to get the answer and judge it
import json
import pandas as pd


def run_judge(question: str, rag_output: dict, search_type: str) -> dict:
    """Judge one answer against exactly the context used to produce it."""

    return rag.evaluate_relevance(
        question=question,
        answer=rag_output["answer"],
        context=rag_output["context"],
        search_type=search_type,
        model="gpt-5.4-mini",
    )


def judge_fields(judge_output: dict) -> dict:
    """Extract the normalized evaluation fields for a table."""

    evaluation = judge_output["evaluation"]

    return {
        "relevance_score": evaluation.get("relevance_score"),
        "groundedness_score": evaluation.get("groundedness_score"),
        "overall_score": evaluation.get("overall_score"),
        "relevance_label": evaluation.get("relevance_label"),
        "groundedness_label": evaluation.get("groundedness_label"),
        "explanation": evaluation.get("explanation"),
    }


def print_comparison_result(
    title: str,
    question_type: str | None,
    output: dict,
    judge: dict,
) -> None:
    """Print the answer and its judge result in a readable notebook format."""

    # evaluation = judge["evaluation"]
    evaluation = judge.get("evaluation", judge)

    print(f"\n{'=' * 100}")
    print(title)
    print(f"Question type: {question_type or 'None — generic baseline'}")
    print("=" * 100)

    print("\n--- Answer ---")
    print(output["answer"])

    print("\n--- Judge evaluation ---")
    print(json.dumps(evaluation, indent=2))

    print("\n--- Generation usage ---")
    print(
        f"Input tokens: {output.get('input_tokens', 0):,} | "
        f"Output tokens: {output.get('output_tokens', 0):,} | "
        f"Cost: ${output.get('total_cost', 0.0):.6f}"
    )

    print("\n--- Judge usage ---")
    print(
        f"Input tokens: {judge['token_stats'].get('input_tokens', 0):,} | "
        f"Output tokens: {judge['token_stats'].get('output_tokens', 0):,} | "
        f"Cost: ${judge['cost'].get('total_cost', 0.0):.6f}"
    )

    print("\n--- Context size ---")
    print(f"{len(output['context']):,} characters")

In [41]:
from question_router import route_question

condition_question = (
    "What are this patient's main diagnosed conditions and their status?"
)

route = route_question(condition_question)

print("=== Router output ===")
print(f"Suggested question type: {route.question_type}")
print(f"Confidence: {route.confidence:.0%}")
print(json.dumps(route.scores, indent=2))


routed_out = rag.rag_new(
    query=condition_question,
    patient_id=patient_id,
    question_type=route.question_type,
    num_results=num_results,
    model=model,
    search_type=search_type,
)

routed_judge = run_judge(
    question=condition_question,
    rag_output=routed_out,
    search_type=search_type,
)

print_comparison_result(
    title="WITH QUESTION ROUTING",
    question_type=route.question_type,
    output=routed_out,
    judge=routed_judge,
)

=== Router output ===
Suggested question type: conditions
Confidence: 100%
{
  "patient_overview": 0.5,
  "conditions": 7.0,
  "medications": 0.0,
  "oncology_timeline": 0.0
}

WITH QUESTION ROUTING
Question type: conditions

--- Answer ---
**Diagnoses and disorders**
- Hypoxemia — active; date: 2020-05-04.
- Malignant neoplasm of breast — active; date: 2015-05-07.
- Viral sinusitis — resolved; date: 1967-05-04.
- Acute pulmonary embolism — resolved; date: 2020-05-11.
- Pneumonia — resolved; date: 2020-05-04.
- Sepsis caused by virus — resolved; date: 2020-05-04.
- Sprain of wrist — resolved; date: 2012-09-08.
- Otitis media — resolved; date: 1965-08-03.
- Injury of medial collateral ligament of knee — resolved; date: 1979-03-02.
- Normal pregnancy — resolved; date: 1995-11-21.
- Acute bronchitis — resolved; date: 1997-09-23.

**Findings and social/functional history**
- Concussion with loss of consciousness — resolved; date: 2022-04-22.
- Stress (finding) — active; date: 2022-02-15.
-

### Example patient without question routing

We get a neutral baseline because `prompt_mode` defaults to "summary" and retrieval covers over all document types.

The judge is happy (relevance_score is 2, groundedness_score: 1, overall_score: 1.5, RELEVANT, PARTLY_GROUNDED), but the answer completely misses the cancer and other active diagnosis.

In [42]:
unrouted_out = rag.rag_new(
    query=condition_question,
    patient_id=patient_id,
    question_type=None,
    num_results=num_results,
    model=model,
    search_type=search_type,
)

unrouted_judge = run_judge(
    question=condition_question,
    rag_output=unrouted_out,
    search_type=search_type,
)

print_comparison_result(
    title="WITHOUT QUESTION ROUTING",
    question_type=None,
    output=unrouted_out,
    judge=unrouted_judge,
)


WITHOUT QUESTION ROUTING
Question type: None — generic baseline

--- Answer ---
1. **Summary:**
- The documented clinical conditions include a concussion with loss of consciousness, hypoxemia, acute pulmonary embolism, pneumonia, sepsis caused by virus, viral sinusitis, and remote normal pregnancy and acute viral pharyngitis; stress and full-time employment are also listed in the record but are not clinical diagnoses.
- The most recent clinically important status is that the concussion with loss of consciousness was resolved by 2022-06-29, while hypoxemia remains active from 2020-05-04 and several prior acute illnesses are resolved.
- No oncology timeline information is documented in the provided context.

2. **Active Conditions:**
- **Hypoxemia** — active; date: 2020-05-04
- **Stress (finding)** — active; date: 2022-02-15

3. **Medications:**
No current medication is documented in the provided medication snapshot.

4. **Oncology timeline:**
No oncology timeline information documented